# Fase 6: Manejo del desbalance del dataset

En esta fase se aplican estrategias para mejorar la detección de la clase minoritaria del problema, es decir, los pacientes que no asisten a su cita médica.

## Objetivo de la fase

El objetivo de esta fase es corregir el principal problema detectado en etapas anteriores: el bajo recall de la clase `No_show`.

Para ello, se utilizarán técnicas de entrenamiento orientadas a datasets desbalanceados, priorizando métricas como recall y f1-score sobre accuracy.

In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [17]:
path = r"C:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\data\KaggleV2-May-2016-features.csv"

df = pd.read_csv(path)

print("Dataset con features cargado correctamente")
print(df.shape)
df.head()

Dataset con features cargado correctamente
(110526, 22)


,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,...,waiting_days,scheduled_weekday,appointment_weekday,is_same_day,age_group,has_comorbidity,risk_score,is_child_or_senior,appointment_month,schedule_hour
0,0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,0,1,0,0,0,...,0,4,4,1,senior,1,1,1,4,18
1,1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,0,0,0,0,...,0,4,4,1,adult,0,0,0,4,16
2,0,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,0,0,0,0,0,...,0,4,4,1,senior,0,0,1,4,16
3,0,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,0,0,0,0,0,...,0,4,4,1,child,0,0,1,4,17
4,0,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,1,1,0,0,...,0,4,4,1,adult,1,2,0,4,16


In [18]:
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'], errors='coerce')
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'], errors='coerce')

## Definición de variables de entrada y salida

Se separa la variable objetivo y se eliminan columnas que no serán utilizadas directamente por el modelo.

In [19]:
y = df['No_show']

X = df.drop(columns=[
    'No_show',
    'ScheduledDay',
    'AppointmentDay'
])

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

Shape de X: (110526, 19)
Shape de y: (110526,)


In [20]:
categorical_cols = ['Neighbourhood', 'age_group']
numeric_cols = [col for col in X.columns if col not in categorical_cols]

print("Variables categóricas:", categorical_cols)
print("Variables numéricas:", numeric_cols)

Variables categóricas: ['Neighbourhood', 'age_group']
Variables numéricas: ['Gender', 'Age', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'waiting_days', 'scheduled_weekday', 'appointment_weekday', 'is_same_day', 'has_comorbidity', 'risk_score', 'is_child_or_senior', 'appointment_month', 'schedule_hour']


In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

print("\nDistribución train:")
print(y_train.value_counts(normalize=True))

print("\nDistribución test:")
print(y_test.value_counts(normalize=True))

Train size: (88420, 19)
Test size: (22106, 19)

Distribución train:
No_show
0    0.798066
1    0.201934
Name: proportion, dtype: float64

Distribución test:
No_show
0    0.798064
1    0.201936
Name: proportion, dtype: float64


## Modelos con manejo de desbalance

En esta fase se probarán modelos con `class_weight='balanced'` cuando sea posible.

In [23]:
balanced_models = {
    "Logistic Regression Balanced": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Decision Tree Balanced": DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "Random Forest Balanced": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    )
}

In [24]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else None

    results = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan,
        "conf_matrix": confusion_matrix(y_test, y_pred),
        "report": classification_report(y_test, y_pred, zero_division=0)
    }

    return results

In [25]:
balanced_results = []

for name, model in balanced_models.items():
    print(f"\nEntrenando: {name}")

    results = evaluate_model(model, X_train, X_test, y_train, y_test)

    balanced_results.append({
        "model": name,
        "accuracy": results["accuracy"],
        "precision": results["precision"],
        "recall": results["recall"],
        "f1": results["f1"],
        "roc_auc": results["roc_auc"]
    })

    print(results["report"])
    print("Confusion Matrix:\n", results["conf_matrix"])


Entrenando: Logistic Regression Balanced
              precision    recall  f1-score   support

           0       0.93      0.51      0.66     17642
           1       0.30      0.84      0.45      4464

    accuracy                           0.58     22106
   macro avg       0.62      0.68      0.55     22106
weighted avg       0.80      0.58      0.62     22106

Confusion Matrix:
 [[8997 8645]
 [ 696 3768]]

Entrenando: Decision Tree Balanced
              precision    recall  f1-score   support

           0       0.83      0.81      0.82     17642
           1       0.32      0.36      0.34      4464

    accuracy                           0.72     22106
   macro avg       0.58      0.59      0.58     22106
weighted avg       0.73      0.72      0.72     22106

Confusion Matrix:
 [[14239  3403]
 [ 2837  1627]]

Entrenando: Random Forest Balanced
              precision    recall  f1-score   support

           0       0.82      0.96      0.88     17642
           1       0.49    

In [26]:
balanced_results_df = pd.DataFrame(balanced_results)
balanced_results_df = balanced_results_df.sort_values(by='f1', ascending=False)

balanced_results_df

,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression Balanced,0.577445,0.303553,0.844086,0.446525,0.722311
1,Decision Tree Balanced,0.717724,0.323459,0.364471,0.342743,0.585821
2,Random Forest Balanced,0.796707,0.489255,0.153002,0.233106,0.741177


## Comparación con la fase anterior

En esta fase se espera que el recall de la clase minoritaria mejore respecto a los resultados obtenidos en la Fase 5, aunque esto pueda implicar una reducción en accuracy global.

In [27]:
fase5_results = pd.DataFrame([
    {"model": "Decision Tree", "accuracy": 0.732290, "precision": 0.333791, "recall": 0.327061, "f1": 0.330391, "roc_auc": 0.582263},
    {"model": "MLPClassifier", "accuracy": 0.757939, "precision": 0.361363, "recall": 0.258961, "f1": 0.301710, "roc_auc": 0.699334},
    {"model": "Random Forest", "accuracy": 0.797204, "precision": 0.493141, "recall": 0.153002, "f1": 0.233544, "roc_auc": 0.741067},
    {"model": "Logistic Regression", "accuracy": 0.797702, "precision": 0.452381, "recall": 0.008513, "f1": 0.016711, "roc_auc": 0.723446},
    {"model": "Gradient Boosting", "accuracy": 0.797973, "precision": 0.470588, "recall": 0.003584, "f1": 0.007114, "roc_auc": 0.734285}
])

fase5_results

,model,accuracy,precision,recall,f1,roc_auc
0,Decision Tree,0.732290,0.333791,0.327061,0.330391,0.582263
1,MLPClassifier,0.757939,0.361363,0.258961,0.301710,0.699334
2,Random Forest,0.797204,0.493141,0.153002,0.233544,0.741067
3,Logistic Regression,0.797702,0.452381,0.008513,0.016711,0.723446
4,Gradient Boosting,0.797973,0.470588,0.003584,0.007114,0.734285


In [28]:
print("Resultados Fase 5:")
display(fase5_results.sort_values(by='f1', ascending=False))

print("Resultados Fase 6:")
display(balanced_results_df)

Resultados Fase 5:


,model,accuracy,precision,recall,f1,roc_auc
0,Decision Tree,0.732290,0.333791,0.327061,0.330391,0.582263
1,MLPClassifier,0.757939,0.361363,0.258961,0.301710,0.699334
2,Random Forest,0.797204,0.493141,0.153002,0.233544,0.741067
3,Logistic Regression,0.797702,0.452381,0.008513,0.016711,0.723446
4,Gradient Boosting,0.797973,0.470588,0.003584,0.007114,0.734285


Resultados Fase 6:


,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression Balanced,0.577445,0.303553,0.844086,0.446525,0.722311
1,Decision Tree Balanced,0.717724,0.323459,0.364471,0.342743,0.585821
2,Random Forest Balanced,0.796707,0.489255,0.153002,0.233106,0.741177


## Prueba opcional con SMOTE

Si la librería `imbalanced-learn` está instalada, se probará SMOTE para generar un entrenamiento balanceado mediante sobremuestreo de la clase minoritaria.

In [29]:
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    smote_available = True
    print("SMOTE disponible")
except ImportError:
    smote_available = False
    print("SMOTE no está instalado. Puedes instalarlo con: pip install imbalanced-learn")

SMOTE disponible


In [30]:
if smote_available:
    smote_model = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('model', LogisticRegression(max_iter=1000))
    ])

    smote_model.fit(X_train, y_train)

    y_pred_smote = smote_model.predict(X_test)
    y_prob_smote = smote_model.predict_proba(X_test)[:, 1]

    smote_results = {
        "model": "Logistic Regression + SMOTE",
        "accuracy": accuracy_score(y_test, y_pred_smote),
        "precision": precision_score(y_test, y_pred_smote, zero_division=0),
        "recall": recall_score(y_test, y_pred_smote, zero_division=0),
        "f1": f1_score(y_test, y_pred_smote, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob_smote)
    }

    print(classification_report(y_test, y_pred_smote, zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_smote))

    smote_results_df = pd.DataFrame([smote_results])
    display(smote_results_df)

              precision    recall  f1-score   support

           0       0.93      0.51      0.66     17642
           1       0.30      0.84      0.45      4464

    accuracy                           0.58     22106
   macro avg       0.62      0.68      0.55     22106
weighted avg       0.80      0.58      0.61     22106

Confusion Matrix:
 [[8974 8668]
 [ 701 3763]]


,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression + SMOTE,0.576178,0.302711,0.842966,0.445457,0.720347


## Interpretación de resultados

En problemas desbalanceados, una disminución en accuracy no necesariamente representa un peor modelo.

Si el recall de la clase minoritaria mejora, el modelo puede resultar mucho más útil para el objetivo real del proyecto: identificar pacientes con riesgo de inasistencia.

## Conclusión técnica de la Fase 6

El manejo del desbalance permitió evaluar modelos con un enfoque más realista sobre el problema.

En lugar de priorizar únicamente accuracy, se dio mayor importancia a la capacidad de detectar pacientes que no asistirán a su cita, usando recall y f1-score como métricas principales.

Esto representa una mejora metodológica importante respecto a la fase anterior.

## Selección del modelo final

En la fase anterior, los modelos con mayor accuracy presentaban un desempeño deficiente en la detección de la clase minoritaria (pacientes que no asisten).

En esta fase se aplicaron técnicas de manejo de desbalance, como `class_weight='balanced'` y SMOTE.

El modelo de regresión logística balanceado logró:

- Recall: 0.84
- F1-score: 0.45

Esto representa una mejora significativa respecto a la fase anterior, donde el recall era cercano a cero en algunos modelos.

Por lo tanto, se selecciona este modelo como el más adecuado, ya que prioriza la detección de pacientes con riesgo de inasistencia, que es el objetivo principal del proyecto.

## Conclusión de la Fase 6

El manejo del desbalance permitió mejorar significativamente la capacidad del modelo para detectar pacientes que no asistirán a su cita.

Aunque esto implicó una reducción en la accuracy global, se logró un aumento considerable en recall y F1-score, métricas más relevantes para este problema.

Este resultado demuestra la importancia de utilizar técnicas adecuadas para datasets desbalanceados y refuerza la validez del enfoque adoptado en el proyecto.